In [1]:
import pandas as pd
import random
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

d:\PROJECT\BookSage AI\book_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0305 13:41:40.539000 33576 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
df = pd.read_pickle("dataset\\cleaned_book_dataset.pkl")

In [3]:
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

d:\PROJECT\BookSage AI\book_venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
train_examples = []

In [5]:
for i in range(len(df)):
    anchor = df.iloc[i]["document"]

    same_genre = df[df["Main Genre"] == df.iloc[i]["Main Genre"]]
    if len(same_genre) > 1:
        positive = same_genre.sample(1).iloc[0]["document"]
    else:
        continue

    diff_genre = df[df["Main Genre"] != df.iloc[i]["Main Genre"]]
    negative = diff_genre.sample(1).iloc[0]["document"]

    train_examples.append(InputExample(texts=[anchor, positive, negative]))

print("Total training samples:", len(train_examples))

Total training samples: 5259


In [6]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.TripletLoss(model)

In [7]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    show_progress_bar=True
)

Epoch: 100%|██████████| 3/3 [06:25<00:00, 128.44s/it]


In [8]:
model.save("models\\custom_embedding_model")